# Pandas sur les données météorologiques du Max Planck Institute for Biogeochemistry

Crédit : certains morceaux de code de ces travaux pratiques sont adaptés depuis l'excellent [tutoriel TensorFlow sur les séries temporelles](https://www.tensorflow.org/tutorials/structured_data/time_series), dont le code est sous licence Apache-2.0. C'est une excellente ressource que vous pouvez consulter si vous souhaiter étudier comment traiter des séries temporelles avec des réseaux de neurones.

## Imports

In [ ]:
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
import tensorflow as tf

## Import de `pandas`

Importez `pandas` de la manière standard vue en cours.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import pandas as pd

## Récupération d'un jeu de données

In [ ]:
zip_path = tf.keras.utils.get_file(
  origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip",
  fname="jena_climate_2009_2016.csv.zip",
  extract=True,
)
csv_path, _ = os.path.splitext(zip_path)
csv_path += "/jena_climate_2009_2016.csv"

## Lecture du fichier d'entrée

Utilisez `pandas.read_csv` pour lire le fichier d'entrée (stocké au chemin `csv_path`) dans la `DataFrame` `df`.

Vous devrez faire attention à ce que les dates soient parsées, et que la colonne d'index soit `Date Time`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
df = pd.read_csv(
  csv_path, parse_dates=True, date_format="%d.%m.%Y %H:%M:%S", index_col="Date Time"
)
df

## Étude de `df`

Étudiez `df` avec les moyens vus en cours. Déterminez :

- Les variables de l'ensemble de données (les colonnes)
- Le nombre d'échantillons
- Le temps entre deux échantillons
- Les caractéristiques des variables numériques

In [ ]:
# Votre code ici

### Solution

In [ ]:
print(f"Forme de la DataFrame : {df.shape}")
print(f"Colonnes de la DataFrame : {', '.join(df.columns)}")
print(f"Temps entre deux échantillons : {df.index[1] - df.index[0]}")
df.describe().transpose()

## Corriger les valeurs aberrantes

Comme on peut le constater ci-dessus, deux colonnes ont des valeurs de vitesse de vent à `-9999.0`, ce qui est bien sûr aberrant.

Corrigez ces valeurs en les remplaçant par `0`.

Vérifiez ensuite avec un appel à `describe` que la correction a bien été appliquée.

In [ ]:
# Votre code ici

### Solution

In [ ]:
wv = df["wv (m/s)"]
mask = df["wv (m/s)"] == -9999.0
wv[mask] = 0.0

max_wv = df["max. wv (m/s)"]
mask = max_wv == -9999.0
max_wv[mask] = 0.0

## Rééchantillonnage

Comme nous l'avons vu à la question précédente, l'intervalle de temps entre deux échantillons est pour l'instant très faible.

Rééchantillonnez le jeu de données à l'aide de la fonction [`pandas.DataFrame.resample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html) pour que les mesures soient moyennées par jour.

In [ ]:
# Votre code ici

### Solution

In [ ]:
df_resampled = df.resample("1D").mean()
print(f"Forme de la DataFrame : {df_resampled.shape}")
print(
  f"Temps entre deux échantillons : {df_resampled.index[1] - df_resampled.index[0]}"
)

## Comparaisons

Calculez les moyennes de température et de pression (`T (degC)` et `p (mbar)`) des mois d'avril 2014 et d'avril 2015. Quelle était l'année la plus chaude ? Avec la pression la plus haute ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
def compute_stats(date: str) -> pd.Series:
  return df_resampled.loc[date, ["T (degC)", "p (mbar)"]].mean()


stats_2014_04, stats_2015_04 = compute_stats("2014-04"), compute_stats("2015-04")
stats_2015_04 > stats_2014_04

## Visualisation d'une `DataFrame`

Affichez les colonnes `T (degC)` & `p (mbar)` grâce à la méthode [`pandas.DataFrame.plot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html). Vous pourrez utiliser l'argument `subplots=True` dans votre appel à `plot` pour rendre le résultat plus lisible.

Le but est de reproduire plus succintement le résultat suivant :

In [ ]:
_, (ax_t, ax_p) = plt.subplots(2, sharex=True)


def plot_column(ax: mpl.axes.Axes, column: str, color: str, xlabel: bool) -> None:
  x = df_resampled.index
  ax.plot(x, df_resampled[column], label=column, color=color)
  ax.set_xlim(x.min(), x.max())
  if xlabel:
    ax.set_xlabel(df_resampled.index.name)
  ax.legend()


plot_column(ax_t, "T (degC)", color="C0", xlabel=False)
plot_column(ax_p, "p (mbar)", color="C1", xlabel=True)
plt.show()

In [ ]:
# Votre code ici

### Solution

In [ ]:
df_resampled[["T (degC)", "p (mbar)"]].plot(subplots=True)
plt.show()

# Sur un mois en particulier
df_resampled.loc["2014-01", ["T (degC)", "p (mbar)"]].plot(subplots=True)
plt.show()